# 🍏 Health & Fitness Agent with Bing Grounding 🍎

This notebook creates a Microsoft Foundry prompt agent with `BingGroundingTool`, asks current health and fitness questions through the Responses API, and displays URL citations.

> **Health disclaimer:** The sample is for general educational purposes and is not a substitute for professional medical advice.

## Prerequisites

- Complete [1-basics.ipynb](1-basics.ipynb).
- Create a Grounding with Bing connection in the Foundry project.
- Configure `AI_FOUNDRY_PROJECT_ENDPOINT`, `MODEL_DEPLOYMENT_NAME`, and `GROUNDING_WITH_BING_CONNECTION_NAME` in the root `.env`.
- Review the [Grounding with Bing terms of use](https://www.microsoft.com/bing/apis/grounding-legal-enterprise).

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

## 1. Initial Setup
We'll load environment variables from `.env` and initialize our **AIProjectClient** to manage agents.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    BingGroundingSearchConfiguration,
    BingGroundingSearchToolParameters,
    BingGroundingTool,
    PromptAgentDefinition,
)

env_path = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError(
        "Could not find .env. Complete Lab 00 and place it in the repository root."
    )

load_dotenv(env_path)
tenant_id = os.environ.get("TENANT_ID")
ai_foundry_project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.environ.get("MODEL_DEPLOYMENT_NAME")
missing_variables = [
    name
    for name, value in {
        "TENANT_ID": tenant_id,
        "AI_FOUNDRY_PROJECT_ENDPOINT": ai_foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment_name,
    }.items()
    if not value
]
if missing_variables:
    raise ValueError(f"Missing required .env variables: {', '.join(missing_variables)}")

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(
    endpoint=ai_foundry_project_endpoint,
    credential=credential,
)
openai_client = project_client.get_openai_client()
print(f"📁 Environment loaded from: {env_path}")
print("✅ Successfully initialized AIProjectClient and OpenAI client")

## 2. Create Bing-Grounded Agent 🌐

Retrieve the named project connection, configure the current Bing grounding model types, and create a versioned prompt agent.

In [ ]:
def create_bing_grounded_agent():
    connection_name = os.environ.get("GROUNDING_WITH_BING_CONNECTION_NAME")
    if not connection_name:
        raise EnvironmentError(
            "Set GROUNDING_WITH_BING_CONNECTION_NAME in the root .env file."
        )

    connection = project_client.connections.get(connection_name)
    print(f"🔗 Bing connection ID: {connection.id}")

    bing_tool = BingGroundingTool(
        bing_grounding=BingGroundingSearchToolParameters(
            search_configurations=[
                BingGroundingSearchConfiguration(
                    project_connection_id=connection.id
                )
            ]
        )
    )
    agent = project_client.agents.create_version(
        agent_name="health-bing-agent",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="""
            You are a health and fitness assistant with Bing grounding.
            Use Bing when current information is useful, cite credible sources, state
            that you are not a medical professional, and encourage consultation with
            qualified healthcare professionals.
            """,
            tools=[bing_tool],
        ),
    )
    print(f"🎉 Created agent {agent.name}, version: {agent.version}")
    return agent


bing_agent = create_bing_grounded_agent()

## 3. Ask Current-Information Questions 💬

Use a fresh Conversation for each independent question, invoke the agent through Responses, and keep each response for citation inspection.

In [ ]:
bing_responses = []
bing_conversations = []
questions = [
    "What are some new HIIT workout trends I should know about?",
    "What is the current WHO recommendation for sugar intake?",
    "What recent evidence is available about intermittent fasting for weight management?",
]

for question in questions:
    conversation = openai_client.conversations.create()
    bing_conversations.append(conversation)
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=question,
        extra_body={
            "agent_reference": {
                "name": bing_agent.name,
                "type": "agent_reference",
            }
        },
    )
    bing_responses.append((question, response))
    print(f"✅ Response created for: {question}")

## 4. View Bing-Grounded Answers & Citations

Grounded sources are returned as `url_citation` annotations. Display the title and URL for every cited source.

In [ ]:
def view_bing_response(question, response):
    print("\n" + "=" * 80)
    print(f"USER: {question}")
    print(f"ASSISTANT: {response.output_text}")

    citations = []
    for item in response.output:
        if item.type != "message":
            continue
        for block in item.content:
            if block.type != "output_text":
                continue
            for annotation in block.annotations:
                if annotation.type == "url_citation":
                    citations.append((annotation.title, annotation.url))

    if citations:
        print("🔗 Sources:")
        for title, url in citations:
            print(f"- {title}: {url}")
    else:
        print("ℹ️ No URL citations were returned for this response.")


for question, response in bing_responses:
    view_bing_response(question, response)

## 5. Cleanup & Best Practices
You can optionally delete the agent once you're done. In production, you might keep it around for repeated usage.

### Best Practices
1. **Accuracy** – Bing search results may include disclaimers or partial info. Encourage verification with credible sources.
2. **Bing Query Display** – For compliance with Bing's use and display requirements, show both **website URLs** (in the agent's response) and **Bing search query URLs** (shown above). If the model includes citations, display them as well.
3. **Limits** – Keep an eye on usage, rate limits, or policy constraints for Bing.
4. **Privacy** – Filter search queries to avoid sending sensitive data.
5. **Evaluations** – Use `azure-ai-evaluation` for iterative improvement.


In [ ]:
for conversation in bing_conversations:
    openai_client.conversations.delete(conversation_id=conversation.id)
print("🗑️ Deleted Conversations.")

project_client.agents.delete_version(
    agent_name=bing_agent.name,
    agent_version=bing_agent.version,
)
print("🗑️ Deleted Bing-grounded agent version.")

openai_client.close()
project_client.close()
credential.close()
print("✅ Cleanup completed!")

# Congratulations! 🎉

You retrieved a Foundry project connection, configured `BingGroundingTool` with `BingGroundingSearchToolParameters`, invoked a versioned prompt agent through Responses, and displayed `url_citation` annotations. Continue to protect sensitive query data, observe Bing usage terms, and retain the health disclaimer.